# Complexity-conditioned 6/20 real diagnostic

> **历史诊断归档：禁止恢复或重新运行。** 本 Notebook 依赖已移除的 anchor/checkpoint v5 架构，仅保留 depth=6 诊断证据；旧模型、optimizer、scheduler、calibration与checkpoint均不得进入新的 no-anchor run。

独立新 run：`max_depth=6, max_nodes=20`。本 Notebook 只使用 training-only RealReward；不会加载 validation/OOS，不会恢复 6/15 或 scalar-logZ checkpoint，也不会自动修改搜索边界。

`calibration=16/32`、`same-N retry=2`、`anchor_frequency=8` 和 64–128 successful updates 均仅为本次 diagnostic 工作参数，不是正式 Stage 5 冻结值。

In [1]:
import json
import platform
import sys
from dataclasses import asdict
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from IPython.display import display

working_dir = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (working_dir, *working_dir.parents) if (p / 'factor_gfn').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('无法从当前目录向上找到 factor_gfn 项目根目录')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from factor_gfn.barra import STYLE_NAMES
from factor_gfn.gfn import (
    DEFAULT_REAL_REWARD_CONFIG, ComplexitySchedulerConfig,
    ConditionalDiagnosticRunSettings, ConditionalDiagnosticRunner,
    DepthBoundaryDiagnosticConfig, DiagnosticRewardProvider,
    ExhaustiveAnchorConfig, ExhaustiveAnchorPool, ExhaustivePlanningConfig,
    ExhaustiveRegistry, GFNConfig, GFNTrainer, ModelConfig,
    NormalizerCalibrationConfig, RealRewardDataPaths, RealRewardProvider,
    SamplingConfig, TrainingConfig, build_real_reward_data_context,
    resolve_exhaustive_plan,
)
from factor_gfn.grammar import Expression, SearchSpaceConfig, resolve_exact_node_strata

## 只读预检
确认数据路径、GPU 和本次工作参数。此单元不创建 run、不做 RealReward 评价。

In [3]:
RUN_DIAGNOSTIC = True  # 确认本单元输出后，人工改为 True
DEVICE = 'cuda:0'
SEED = 42
RUN_NAME = 'manual_diagnostic_6_20_seed42'
MAX_DEPTH = 6
MAX_NODES = 20
BATCH_SIZE = 8
EXACT_NODE_RETRY_BUDGET = 2
CALIBRATION_MIN_VALID_PER_N = 16
CALIBRATION_MAX_REQUESTED_PER_N = 32
ANCHOR_FREQUENCY = 8
ANCHOR_BATCH_SIZE = 8
MIN_SUCCESSFUL_UPDATES = 64
MAX_SUCCESSFUL_UPDATES = 128
MAX_LOGICAL_BATCHES = 160
BOUNDARY_CHECK_INTERVAL = 8
CHECKPOINT_INTERVAL = 8
RUN_ROOT = PROJECT_ROOT / 'runs' / 'complexity_diagnostic_6_20' / RUN_NAME
REGISTRY_PATH = RUN_ROOT / 'exhaustive_registry.sqlite3'
AUDIT_PATH = RUN_ROOT / 'candidate_audit.jsonl'

if not DEVICE.startswith('cuda:'):
    raise ValueError('真实 diagnostic 必须显式使用 CUDA device')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA 不可用；不要回落 CPU 执行真实 diagnostic')
device = torch.device(DEVICE)
torch.cuda.set_device(device)
data_paths = RealRewardDataPaths()
barra_paths = data_paths.barra_paths
required_paths = [
    data_paths.tensor_path, data_paths.universe_mask_path, data_paths.date_list_path,
    data_paths.stock_list_path, data_paths.processed_metadata_path,
    data_paths.industry_path, data_paths.industry_metadata_path,
    barra_paths.metadata_path, barra_paths.market_return_path,
    *[barra_paths.exposure_path(name) for name in STYLE_NAMES],
]
input_frame = pd.DataFrame([{'path': str(p), 'exists': p.is_file(), 'size_mib': p.stat().st_size / 1024**2 if p.is_file() else np.nan} for p in required_paths])
display(input_frame)
missing = input_frame.loc[~input_frame['exists'], 'path'].tolist()
if missing:
    raise FileNotFoundError(f'真实 Reward 输入缺失：{missing}')
search_space = SearchSpaceConfig(max_depth=MAX_DEPTH, max_nodes=MAX_NODES)
static_strata = resolve_exact_node_strata(search_space)
display(pd.DataFrame([{
    'project_root': str(PROJECT_ROOT), 'run_root': str(RUN_ROOT),
    'python': platform.python_version(), 'torch': torch.__version__,
    'device': str(device), 'gpu': torch.cuda.get_device_name(device),
    'max_depth': MAX_DEPTH, 'max_nodes': MAX_NODES,
    'static_F': static_strata.resolved_feasible_node_counts,
    'static_infeasible': static_strata.resolved_infeasible_node_counts,
    'search_space_fingerprint': search_space.fingerprint(),
    'run_enabled': RUN_DIAGNOSTIC,
}]))
print('本次参数仅用于 6/20 diagnostic，不冻结正式 Stage 5 参数。')
print('确认路径、GPU、6/20 边界后，再将 RUN_DIAGNOSTIC 改为 True。')

,path,exists,size_mib
0,D:\实习\Gflownet因子挖掘\data\processed\data_tensor.npy,True,499.934082
1,D:\实习\Gflownet因子挖掘\data\processed\universe_mas...,True,20.830704
2,D:\实习\Gflownet因子挖掘\data\processed\date_list.npy,True,0.030846
3,D:\实习\Gflownet因子挖掘\data\processed\stock_list.npy,True,0.124268
4,D:\实习\Gflownet因子挖掘\data\processed\metadata.json,True,0.001636
5,D:\实习\Gflownet因子挖掘\data\processed\industry_sw_...,True,28.786022
6,D:\实习\Gflownet因子挖掘\data\processed\industry_sw_...,True,0.003881
7,D:\实习\Gflownet因子挖掘\data\processed\barra\metada...,True,0.001117
8,D:\实习\Gflownet因子挖掘\data\processed\barra\market...,True,0.030846
9,D:\实习\Gflownet因子挖掘\data\processed\barra\market...,True,83.322449


,project_root,run_root,python,torch,device,gpu,max_depth,max_nodes,static_F,static_infeasible,search_space_fingerprint,run_enabled
0,D:\实习\Gflownet因子挖掘,D:\实习\Gflownet因子挖掘\runs\complexity_diagnostic_...,3.12.13,2.6.0+cu124,cuda:0,NVIDIA GeForce RTX 4060 Laptop GPU,6,20,"(1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",(),2857d50cb9d85cae5881c46fd25f255a6b9780e5781d7b...,True


本次参数仅用于 6/20 diagnostic，不冻结正式 Stage 5 参数。
确认路径、GPU、6/20 边界后，再将 RUN_DIAGNOSTIC 改为 True。


## 解析 F，并复用已验证的 E=(1,2)
6/15真实smoke已完整确认N=1有6个、N=2有636个canonical terminals，N=3超过自动exhaustive阈值。本次6/20不重复枚举N=3…20：只复核N=1/2，其他可达层显式保持discovery，并将这一诊断决策写入配置与run指纹。

In [4]:
if not RUN_DIAGNOSTIC:
    raise RuntimeError('安全锁仍为 False；确认重大参数后再人工开启')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
phase_times = {}
total_started = perf_counter()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)

started = perf_counter()
context = build_real_reward_data_context()
base_provider = RealRewardProvider(context, DEFAULT_REAL_REWARD_CONFIG)
provider = DiagnosticRewardProvider(base_provider, AUDIT_PATH)
phase_times['real_reward_context_and_provider'] = perf_counter() - started
provider_manifest = provider.manifest()
assert provider_manifest['data_scope'] == 'training_only'
assert provider_manifest['validation_oos_loaded'] is False
assert provider_manifest['industry_neutralization']['enabled'] is True
assert base_provider.reward_config.candidate_industry_neutralization is True

started = perf_counter()
VERIFIED_EXHAUSTIVE_NODE_COUNTS = (1, 2)
resolved_feasible = static_strata.resolved_feasible_node_counts
if not set(VERIFIED_EXHAUSTIVE_NODE_COUNTS).issubset(resolved_feasible):
    raise RuntimeError('当前6/20搜索空间不再包含已验证的N=1/2；停止并检查Grammar')
verified_discovery_node_counts = tuple(n for n in resolved_feasible if n not in VERIFIED_EXHAUSTIVE_NODE_COUNTS)
planning = ExhaustivePlanningConfig(
    canonical_count_cap=10_000,
    estimated_real_reward_seconds_per_candidate=0.75,
    planned_real_reward_budget_seconds=3600.0,
    max_budget_fraction=0.20,
    explicit_include_node_counts=VERIFIED_EXHAUSTIVE_NODE_COUNTS,
    explicit_exclude_node_counts=verified_discovery_node_counts,
)
plan = resolve_exhaustive_plan(search_space, planning)
phase_times['verified_n1_n2_count_and_strata_resolution'] = perf_counter() - started
if plan.resolved_exhaustive_node_counts != VERIFIED_EXHAUSTIVE_NODE_COUNTS:
    raise RuntimeError(f'解析E={plan.resolved_exhaustive_node_counts}与已验证E={VERIFIED_EXHAUSTIVE_NODE_COUNTS}不一致')
if plan.resolved_discovery_node_counts != verified_discovery_node_counts:
    raise RuntimeError('解析S与显式保留的discovery strata不一致')

config = GFNConfig(
    search_space=search_space,
    model=ModelConfig(d_model=128, num_heads=4, num_layers=4, dim_feedforward=512, dropout=0.0, token_policy_mode='grammar_hierarchical'),
    sampling=SamplingConfig(temperature=1.0, greedy=False),
    reward=DEFAULT_REAL_REWARD_CONFIG,
    training=TrainingConfig(
        batch_size=BATCH_SIZE, learning_rate=1e-4, log_z_learning_rate=1e-2,
        initial_log_z=39.0, max_steps=MAX_LOGICAL_BATCHES,
        model_gradient_clip_norm=5.0, log_z_gradient_clip_norm=5.0,
        deterministic_algorithms=True, seed=SEED,
    ),
    complexity_scheduler=ComplexitySchedulerConfig(
        enabled=True, exhaustive_node_counts=plan.resolved_exhaustive_node_counts,
        exact_node_retry_budget=EXACT_NODE_RETRY_BUDGET,
        low_effective_update_rate_warning_threshold=0.25,
    ),
    calibration=NormalizerCalibrationConfig(
        enabled=True, minimum_valid_calibration_samples=CALIBRATION_MIN_VALID_PER_N,
        maximum_requested_calibration_slots_per_N=CALIBRATION_MAX_REQUESTED_PER_N,
    ),
    exhaustive_anchors=ExhaustiveAnchorConfig(
        enabled=True, frequency=ANCHOR_FREQUENCY, batch_size=ANCHOR_BATCH_SIZE, seed_offset=1,
    ),
)
resolved = config.resolved_complexity_strata()
assert resolved['resolved_feasible_node_counts'] == plan.resolved_feasible_node_counts
assert resolved['resolved_exhaustive_node_counts'] == plan.resolved_exhaustive_node_counts
assert resolved['resolved_discovery_node_counts'] == plan.resolved_discovery_node_counts
display(pd.DataFrame([{
    'F': resolved['resolved_feasible_node_counts'],
    'E': resolved['resolved_exhaustive_node_counts'],
    'S': resolved['resolved_discovery_node_counts'],
    'estimated_exhaustive_seconds': plan.resolved_estimated_evaluation_seconds,
    'cumulative_budget_seconds': plan.exhaustive_budget_seconds,
    'config_fingerprint': config.fingerprint(),
    'provider_fingerprint': provider.fingerprint(),
    'context_fingerprint': context.fingerprint,
}]))
display(pd.DataFrame([item.manifest() for item in plan.count_results]).set_index('node_count'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,F,E,S,estimated_exhaustive_seconds,cumulative_budget_seconds,config_fingerprint,provider_fingerprint,context_fingerprint
0,"(1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","(1, 2)","(3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, ...",481.5,720.0,25d9014fcbf060ab50ef5ae9747938e8b03207081d5dde...,dce89ac7b21afdbeeab1da9d6e29d61094e2d927efa594...,1a2a6c10c6d8ad64d0c6ef4ba7b4029628bd1e87772eb3...


,canonical_terminal_count,canonical_count_exact,count_cap_reached,count_relation,depth_distribution,depth_distribution_exact,estimated_evaluation_seconds,estimated_evaluation_seconds_is_lower_bound
node_count,,,,,,,,
1,6,True,False,=,{0: 6},True,4.5,False
2,636,True,False,=,{1: 636},True,477.0,False


## 可恢复 exhaustive evaluation、exact Z 与 calibration

In [5]:
started = perf_counter()
registry = ExhaustiveRegistry(REGISTRY_PATH)
registry.register_plan(plan, provider_fingerprint=provider.fingerprint(), context_fingerprint=context.fingerprint)
phase_times['exhaustive_registry_registration'] = perf_counter() - started

started = perf_counter()
with provider.phase('exhaustive_full_evaluation'):
    for node_count in plan.resolved_exhaustive_node_counts:
        pending = registry.pending_candidates(node_count)
        print(f'N={node_count}: pending={len(pending)}, coverage={registry.coverage(node_count)}')
        for index, candidate in enumerate(pending, start=1):
            assignment = provider.evaluate(Expression.from_prefix(candidate.prefix_token_ids))
            registry.record_evaluation(
                candidate.structural_hash, valid=assignment.valid,
                reward_details=assignment.metadata or {},
                rejection_reason=assignment.rejection_reason,
                target_mass=assignment.reward if assignment.valid else 0.0,
            )
            if index % 25 == 0 or index == len(pending):
                print({'N': node_count, 'finished_this_run': index, **registry.coverage(node_count)})
phase_times['exhaustive_real_reward'] = perf_counter() - started

exact_results = {n: registry.compute_exact_masses(n, reward_floor=config.reward.reward_floor) for n in plan.resolved_exhaustive_node_counts}
exact_frame = pd.DataFrame([{**asdict(v), **registry.coverage(n)} for n, v in exact_results.items()]).set_index('node_count')
display(exact_frame)
assert exact_frame['coverage_complete'].all()
assert np.isfinite(exact_frame['exact_tb_log_z'].astype(float)).all()

started = perf_counter()
trainer = GFNTrainer(config, provider, device=device)
for result in exact_results.values():
    trainer.register_exact_mass_result(result)
anchor_pool = ExhaustiveAnchorPool.from_registry(registry, node_counts=plan.resolved_exhaustive_node_counts, adapter=trainer.adapter)
trainer.configure_exhaustive_anchor_pool(anchor_pool)
phase_times['trainer_and_anchor_pool'] = perf_counter() - started

settings = ConditionalDiagnosticRunSettings(
    minimum_successful_discovery_updates=MIN_SUCCESSFUL_UPDATES,
    maximum_successful_discovery_updates=MAX_SUCCESSFUL_UPDATES,
    maximum_logical_batches=MAX_LOGICAL_BATCHES,
    boundary_check_interval_successful_updates=BOUNDARY_CHECK_INTERVAL,
    checkpoint_interval_logical_batches=CHECKPOINT_INTERVAL,
)
depth_config = DepthBoundaryDiagnosticConfig()
checkpoint_path = RUN_ROOT / 'diagnostic_checkpoint.pt'
if checkpoint_path.exists():
    runner = ConditionalDiagnosticRunner(trainer, provider, RUN_ROOT, settings=settings, depth_config=depth_config)
    assert runner.resume_if_available()
    print(f'已恢复：logical_batch={trainer.step}, successful_update={trainer.optimizer_step}')
else:
    started = perf_counter()
    with provider.phase('calibration'):
        calibration_result = None
        while calibration_result is None:
            calibration_result = trainer.calibration_step()
    phase_times['training_only_calibration'] = perf_counter() - started
    runner = ConditionalDiagnosticRunner(trainer, provider, RUN_ROOT, settings=settings, depth_config=depth_config)
calibration_frame = pd.DataFrame([asdict(value) for value in trainer.calibration_report().values()]).set_index('node_count').sort_index()
display(calibration_frame)
assert trainer.calibration.status == 'complete'
assert provider.manifest()['validation_oos_loaded'] is False

N=1: pending=6, coverage={'node_count': 1, 'expected_canonical_count': 6, 'registered_count': 6, 'evaluated_count': 0, 'valid_count': 0, 'invalid_count': 0, 'enumeration_complete': True, 'evaluation_complete': False, 'coverage_complete': False}
{'N': 1, 'finished_this_run': 6, 'node_count': 1, 'expected_canonical_count': 6, 'registered_count': 6, 'evaluated_count': 6, 'valid_count': 6, 'invalid_count': 0, 'enumeration_complete': True, 'evaluation_complete': True, 'coverage_complete': True}
N=2: pending=636, coverage={'node_count': 2, 'expected_canonical_count': 636, 'registered_count': 636, 'evaluated_count': 0, 'valid_count': 0, 'invalid_count': 0, 'enumeration_complete': True, 'evaluation_complete': False, 'coverage_complete': False}
{'N': 2, 'finished_this_run': 25, 'node_count': 2, 'expected_canonical_count': 636, 'registered_count': 636, 'evaluated_count': 25, 'valid_count': 25, 'invalid_count': 0, 'enumeration_complete': True, 'evaluation_complete': False, 'coverage_complete': Fa

,valid_candidate_count,invalid_candidate_count,exact_raw_reward_log_mass,raw_reward_mass_status,exact_tb_log_z,reward_floor,provider_fingerprint,context_fingerprint,aggregation_fingerprint,expected_canonical_count,registered_count,evaluated_count,valid_count,invalid_count,enumeration_complete,evaluation_complete,coverage_complete
node_count,,,,,,,,,,,,,,,,,
1,6,0,-1.766961,positive_mass,-1.766961,1.000000e-08,dce89ac7b21afdbeeab1da9d6e29d61094e2d927efa594...,1a2a6c10c6d8ad64d0c6ef4ba7b4029628bd1e87772eb3...,4d753aa4730ca675f9f7185d3981de816ce5e9cd2d172d...,6,6,6,6,0,True,True,True
2,629,7,2.849463,positive_mass,2.849463,1.000000e-08,dce89ac7b21afdbeeab1da9d6e29d61094e2d927efa594...,1a2a6c10c6d8ad64d0c6ef4ba7b4029628bd1e87772eb3...,a00306d6312e8c73b3bcc2b5c01f23464a9b5db4f5fc16...,636,636,636,629,7,True,True,True


D:\实习\Gflownet因子挖掘\factor_gfn\gfn\reward.py:507: IndustryNeutralizationWarning: 日期索引 22 的行业中性化失败 (reason=insufficient_industry_regression_samples, factor_valid=3, known_industry=3, industries=3)，已排除当日候选截面
  cleaned = clean_candidate_factor_cross_sections(
D:\实习\Gflownet因子挖掘\factor_gfn\gfn\reward.py:507: IndustryNeutralizationWarning: 日期索引 23 的行业中性化失败 (reason=insufficient_industry_regression_samples, factor_valid=3, known_industry=3, industries=3)，已排除当日候选截面
  cleaned = clean_candidate_factor_cross_sections(
D:\实习\Gflownet因子挖掘\factor_gfn\gfn\reward.py:507: IndustryNeutralizationWarning: 日期索引 24 的行业中性化失败 (reason=insufficient_industry_regression_samples, factor_valid=3, known_industry=3, industries=3)，已排除当日候选截面
  cleaned = clean_candidate_factor_cross_sections(
D:\实习\Gflownet因子挖掘\factor_gfn\gfn\reward.py:507: IndustryNeutralizationWarning: 日期索引 25 的行业中性化失败 (reason=insufficient_industry_regression_samples, factor_valid=3, known_industry=3, industries=3)，已排除当日候选截面
  cleaned = clean_candidate

TypeError: asdict() should be called on dataclass instances

In [6]:
trainer.save_checkpoint(RUN_ROOT / 'diagnostic_checkpoint.pt')

calibration_frame = (
    pd.DataFrame(list(trainer.calibration_report().values()))
    .set_index('node_count')
    .sort_index()
)
display(calibration_frame)

print("CALIBRATION_RECOVERED_AND_SAVED")

,calibration_requested,calibration_valid,calibration_sampled_attempts,median,logmeanexp,p10,p25,p75,p90,iqr,median_implied_minus_exact_tb_log_z,logmeanexp_implied_minus_exact_tb_log_z
node_count,,,,,,,,,,,,
1,18,18,18,-2.060616,-1.759581,-2.153706,-2.141618,-2.060231,-0.875903,0.081387,-0.293655,0.007380
2,18,18,18,2.561915,3.218440,1.175245,1.522714,3.845255,4.065534,2.322541,-0.287548,0.368977
3,18,18,20,5.751882,7.553237,3.047633,4.016635,7.779179,8.824964,3.762544,NaN,NaN
4,19,19,23,8.557557,13.427273,6.901747,7.100056,9.847906,12.306595,2.747851,NaN,NaN
5,18,18,21,12.044567,15.464832,9.942926,10.815163,13.818347,16.774343,3.003184,NaN,NaN
6,18,18,21,15.843430,18.432576,11.565595,13.581828,18.650289,19.423654,5.068461,NaN,NaN
7,19,19,29,19.031773,22.502309,17.192020,17.963760,20.387930,22.754697,2.424170,NaN,NaN
8,18,18,24,23.181034,25.919372,20.097746,20.776413,25.291248,26.508874,4.514835,NaN,NaN
9,18,18,21,26.251714,29.507296,22.547504,25.030972,27.982260,29.786467,2.951288,NaN,NaN


CALIBRATION_RECOVERED_AND_SAVED


## 短 conditional training diagnostic
runner 会每 8 个 logical batch 保存 checkpoint。64 次成功 update 只是最早判断点；depth 样本充分性使用审计文件中的实际 `unique_discovery_candidate_count`，绝不使用 update×batch 推算。

In [8]:
started = perf_counter()
summary = runner.run()
phase_times['conditional_training_diagnostic'] = perf_counter() - started
phase_times['total_notebook_wall'] = perf_counter() - total_started
summary['phase_times_this_invocation'] = phase_times
summary['exhaustive_plan'] = plan.manifest()
summary['exact_by_N'] = {str(n): asdict(value) for n, value in exact_results.items()}
summary['calibration_by_N'] = {str(n): asdict(value) for n, value in trainer.calibration_report().items()}
summary['industry_neutralization_on'] = bool(provider.manifest()['industry_neutralization']['enabled'] and base_provider.reward_config.candidate_industry_neutralization)
summary['training_only'] = provider.manifest()['data_scope'] == 'training_only'
summary['validation_oos_not_loaded'] = provider.manifest()['validation_oos_loaded'] is False
(RUN_ROOT / 'diagnostic_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

display(pd.DataFrame([summary]))
display(pd.DataFrame(summary['per_n_training_health']).T.rename_axis('N'))
display(pd.read_csv(RUN_ROOT / 'depth_metrics.csv'))
display(pd.DataFrame([{'phase': name, 'wall_seconds': value} for name, value in phase_times.items()]))
assert summary['resolved_F'] == list(range(1, MAX_NODES + 1)) or tuple(summary['resolved_F']) == tuple(range(1, MAX_NODES + 1))
assert tuple(summary['resolved_E']) == plan.resolved_exhaustive_node_counts
assert tuple(summary['resolved_S']) == plan.resolved_discovery_node_counts
assert summary['unique_discovery_candidate_count'] == runner.provider.unique_discovery_candidate_count()
assert summary['depth_boundary_status'] in {'consider_expansion', 'no_expansion_evidence', 'insufficient_evidence'}
assert summary['industry_neutralization_on']
assert summary['training_only'] and summary['validation_oos_not_loaded']
assert config.search_space.max_depth == MAX_DEPTH and config.search_space.max_nodes == MAX_NODES
registry.close()
print('DIAGNOSTIC_6_20_COMPLETE')
print('结果目录：', RUN_ROOT)
print('停止：不要在原 run 中修改到 7/20 或 8/20；把结果交给 Codex 分析。')

D:\实习\Gflownet因子挖掘\factor_gfn\gfn\reward.py:507: IndustryNeutralizationWarning: 日期索引 22 的行业中性化失败 (reason=insufficient_industry_regression_samples, factor_valid=3, known_industry=3, industries=3)，已排除当日候选截面
  cleaned = clean_candidate_factor_cross_sections(
D:\实习\Gflownet因子挖掘\factor_gfn\gfn\reward.py:507: IndustryNeutralizationWarning: 日期索引 23 的行业中性化失败 (reason=insufficient_industry_regression_samples, factor_valid=3, known_industry=3, industries=3)，已排除当日候选截面
  cleaned = clean_candidate_factor_cross_sections(
D:\实习\Gflownet因子挖掘\factor_gfn\gfn\reward.py:507: IndustryNeutralizationWarning: 日期索引 24 的行业中性化失败 (reason=insufficient_industry_regression_samples, factor_valid=3, known_industry=3, industries=3)，已排除当日候选截面
  cleaned = clean_candidate_factor_cross_sections(
D:\实习\Gflownet因子挖掘\factor_gfn\gfn\reward.py:507: IndustryNeutralizationWarning: 日期索引 25 的行业中性化失败 (reason=insufficient_industry_regression_samples, factor_valid=3, known_industry=3, industries=3)，已排除当日候选截面
  cleaned = clean_candidate

TypeError: asdict() should be called on dataclass instances

In [10]:
summary['calibration_by_N'] = {
    str(n): dict(value)
    for n, value in trainer.calibration_report().items()
}
summary['industry_neutralization_on'] = bool(
    provider.manifest()['industry_neutralization']['enabled']
    and base_provider.reward_config.candidate_industry_neutralization
)
summary['training_only'] = provider.manifest()['data_scope'] == 'training_only'
summary['validation_oos_not_loaded'] = (
    provider.manifest()['validation_oos_loaded'] is False
)

(RUN_ROOT / 'diagnostic_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

display(pd.DataFrame([summary]))
display(pd.DataFrame(summary['per_n_training_health']).T.rename_axis('N'))
display(pd.read_csv(RUN_ROOT / 'depth_metrics.csv'))
display(pd.DataFrame([
    {'phase': name, 'wall_seconds': value}
    for name, value in phase_times.items()
]))

assert summary['resolved_F'] == list(range(1, MAX_NODES + 1)) \
    or tuple(summary['resolved_F']) == tuple(range(1, MAX_NODES + 1))
assert tuple(summary['resolved_E']) == plan.resolved_exhaustive_node_counts
assert tuple(summary['resolved_S']) == plan.resolved_discovery_node_counts
assert (
    summary['unique_discovery_candidate_count']
    == runner.provider.unique_discovery_candidate_count()
)
assert summary['depth_boundary_status'] in {
    'consider_expansion',
    'no_expansion_evidence',
    'insufficient_evidence',
}
assert summary['industry_neutralization_on']
assert summary['training_only']
assert summary['validation_oos_not_loaded']
assert config.search_space.max_depth == MAX_DEPTH
assert config.search_space.max_nodes == MAX_NODES

registry.close()

print('DIAGNOSTIC_6_20_COMPLETE')
print('结果目录：', RUN_ROOT)
print('停止：不要在原 run 中修改到 7/20 或 8/20；把结果交给 Codex 分析。')

,schema,stop_reason,config_fingerprint,resolved_F,resolved_E,resolved_S,logical_batches,successful_discovery_updates,anchor_optimizer_updates,total_policy_optimizer_updates,...,last_step_timings,gpu,work_parameters_are_not_stage5_frozen,phase_times_this_invocation,exhaustive_plan,exact_by_N,calibration_by_N,industry_neutralization_on,training_only,validation_oos_not_loaded
0,factor_gfn.conditional_diagnostic.v1,decisive_depth_boundary_advisory,25d9014fcbf060ab50ef5ae9747938e8b03207081d5dde...,"(1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","(1, 2)","(3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, ...",112,96,12,108,...,"{'sampling_seconds': 35.66549359995406, 'rewar...","{'device': 'cuda:0', 'name': 'NVIDIA GeForce R...",True,{'real_reward_context_and_provider': 11.265080...,{'schema': 'factor_gfn.exhaustive_planning.v1'...,"{'1': {'node_count': 1, 'valid_candidate_count...","{'1': {'node_count': 1, 'calibration_requested...",True,True,True


,requested,sampled_attempts,valid,successful,retry_exhausted,effective_update_rate
N,,,,,,
3,50.0,53.0,50.0,40.0,0.0,0.800000
4,50.0,57.0,50.0,45.0,0.0,0.900000
5,50.0,55.0,50.0,44.0,0.0,0.880000
6,50.0,55.0,50.0,42.0,0.0,0.840000
7,50.0,56.0,49.0,46.0,0.0,0.920000
8,51.0,56.0,49.0,44.0,1.0,0.862745
9,50.0,58.0,49.0,43.0,0.0,0.860000
10,50.0,61.0,48.0,43.0,1.0,0.860000
11,50.0,70.0,48.0,43.0,2.0,0.860000


,scope,node_count,depth,unique_candidate_count,depth_share_within_scope,valid_count,valid_rate,finite_reward_count,reward_p90,reward_p99,finite_abs_ic_count,abs_ic_p90,abs_ic_p99,evaluation_seconds_mean,evaluation_seconds_median
0,by_node_count,3.0,0,0,0.000000,0,NaN,0,NaN,NaN,0,NaN,NaN,NaN,NaN
1,by_node_count,3.0,1,16,0.301887,14,0.875000,14,0.042748,0.058867,14,0.039406,0.040680,0.483148,0.471639
2,by_node_count,3.0,2,37,0.698113,36,0.972973,36,0.061288,0.079992,36,0.048906,0.059703,0.732288,0.663997
3,by_node_count,3.0,3,0,0.000000,0,NaN,0,NaN,NaN,0,NaN,NaN,NaN,NaN
4,by_node_count,3.0,4,0,0.000000,0,NaN,0,NaN,NaN,0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128,overall,NaN,2,107,0.092321,98,0.915888,98,0.052178,0.077555,98,0.043479,0.055694,0.710491,0.681486
129,overall,NaN,3,94,0.081104,86,0.914894,86,0.046169,0.080289,86,0.037285,0.060475,0.918717,0.893320
130,overall,NaN,4,131,0.113028,107,0.816794,107,0.033333,0.069659,107,0.030889,0.056587,1.264552,1.173253
131,overall,NaN,5,150,0.129422,119,0.793333,119,0.025151,0.056691,119,0.021803,0.046094,1.528477,1.484673


,phase,wall_seconds
0,real_reward_context_and_provider,11.265081
1,verified_n1_n2_count_and_strata_resolution,0.488942
2,exhaustive_registry_registration,0.062553
3,exhaustive_real_reward,288.673881
4,trainer_and_anchor_pool,14.500937
5,training_only_calibration,2579.717872
6,conditional_training_diagnostic,7923.257862
7,total_notebook_wall,12955.801823


DIAGNOSTIC_6_20_COMPLETE
结果目录： D:\实习\Gflownet因子挖掘\runs\complexity_diagnostic_6_20\manual_diagnostic_6_20_seed42
停止：不要在原 run 中修改到 7/20 或 8/20；把结果交给 Codex 分析。
